# Sentiment Analysis — Random Forest & GRU
**Julianah | Group K**

This notebook covers two models: a Random Forest classifier using GloVe embeddings + structural metadata, and a Bidirectional GRU neural network. Both are evaluated on the in-domain validation set and the cross-domain student test set (Gmail + WhatsApp).

In [ ]:
import os
import re
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, SpatialDropout1D, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Fix random seeds so results are reproducible each run
np.random.seed(42)
tf.random.set_seed(42)

# Download GloVe 100d vectors from Stanford if not already present
glove_url = "https://nlp.stanford.edu/data/glove.6B.zip"
glove_zip = "glove.6B.zip"

if not os.path.exists('glove.6B.100d.txt'):
    print("Downloading GloVe vectors from Stanford (this takes a few minutes)...")
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(glove_url, glove_zip)
    print("Download done. Extracting...")
    with zipfile.ZipFile(glove_zip, 'r') as z:
        z.extract('glove.6B.100d.txt')
    print("GloVe vectors ready.")
else:
    print("GloVe vectors already downloaded.")

## 1. Load Datasets
We load the training, validation, and cross-domain test sets. The test set contains real Gmail and WhatsApp messages — the hardest part of this evaluation.

In [ ]:
# File names must match exactly what you uploaded to Colab
train_file = 'processed_training_dataset.csv'
val_file   = 'processed_validation_datset.csv'
test_file  = 'student_test_dataset (1).csv'

train_df = pd.read_csv(train_file).dropna(subset=['text', 'sentiment'])
val_df   = pd.read_csv(val_file).dropna(subset=['text', 'sentiment'])
test_df  = pd.read_csv(test_file).dropna(subset=['text', 'sentiment'])

# Map the three sentiment classes to integers
# This is required — over 30% of the test set is Neutral (alerts, invoices, receipts)
label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
train_df['label'] = train_df['sentiment'].map(label_map)
val_df['label']   = val_df['sentiment'].map(label_map)
test_df['label']  = test_df['sentiment'].map(label_map)

print(f"Train: {len(train_df)} samples | Val: {len(val_df)} | Test: {len(test_df)}")
print("\nLabel distribution in test set:")
print(test_df['sentiment'].value_counts())

## 2. Text Preprocessing
A single cleaning function used by both models. Key steps:
- Capture structural metadata (exclamation marks, HTML, ALL CAPS) **before** stripping punctuation
- Remove service prefixes like `MTN:`, `GitHub:`, `Forwarded message`
- Strip email footers, URLs, and @mentions
- Expand contractions (`don't` → `do not`) so classical models see consistent tokens
- Map common WhatsApp shortcuts (`tmrw`, `pls`, `u`) to standard English

In [ ]:
# Contraction expansion dictionary
CONTRACTIONS = {
    "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "won't": "will not", "can't": "cannot", "couldn't": "could not",
    "shouldn't": "should not", "wouldn't": "would not", "isn't": "is not",
    "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "i'm": "i am", "i've": "i have", "i'll": "i will", "i'd": "i would",
    "you're": "you are", "you've": "you have", "you'll": "you will",
    "he's": "he is", "she's": "she is", "it's": "it is",
    "we're": "we are", "we've": "we have", "they're": "they are",
    "that's": "that is", "there's": "there is", "what's": "what is",
    "let's": "let us", "here's": "here is",
}

# WhatsApp / informal shortcut mapping
SLANG_MAP = {
    "\bu\b": "you", "\bur\b": "your", "\br\b": "are",
    "\bpls\b": "please", "\bplz\b": "please",
    "\btmrw\b": "tomorrow", "\btmr\b": "tomorrow",
    "\bwat\b": "what", "\bwht\b": "what",
    "\bthx\b": "thanks", "\bthnks\b": "thanks",
    "\bgr8\b": "great", "\bbtw\b": "by the way",
    "\bidk\b": "i do not know", "\bidc\b": "i do not care",
    "\bngl\b": "not gonna lie", "\bngl\b": "not going to lie",
    "\blmk\b": "let me know", "\bomg\b": "oh my god",
    "\bnp\b": "no problem", "\bok\b": "okay",
    "\bsup\b": "what is up", "\bwbu\b": "what about you",
}

def expand_contractions(text):
    for contraction, expansion in CONTRACTIONS.items():
        text = re.sub(contraction, expansion, text, flags=re.IGNORECASE)
    return text

def apply_slang_map(text):
    for pattern, replacement in SLANG_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def clean_text(df):
    """
    Cleans text and extracts structural metadata features.
    Returns the dataframe with a new 'cleaned_text' column and metadata columns.
    """
    cleaned_texts = []
    meta_features = []

    for text in df['text']:
        t = str(text)

        # --- Step 1: Capture structural signals BEFORE any stripping ---
        exclamation_count  = t.count('!')
        question_count     = t.count('?')
        has_html_artifacts = 1 if bool(re.search(r'<[^>]+>|&amp;|&nbsp;|&lt;|&gt;', t)) else 0
        alpha_only         = re.sub(r'[^a-zA-Z]', '', t)
        is_all_caps        = 1 if (alpha_only.isupper() and len(alpha_only) > 3) else 0
        char_cnt           = len(t)
        word_cnt           = len(t.split())

        # --- Step 2: Remove technical noise (service prefixes, footers, links) ---
        t = re.sub(
            r'^(MTN|GitHub|Forwarded message|Alert|Reminder|Security alert|'
            r'Billing alert|Service termination notice|Coursera|Paystack|DigitalOcean):\s*',
            '', t, flags=re.IGNORECASE
        )
        t = re.sub(r'Please consider the environment before printing.*', '', t, flags=re.IGNORECASE)
        t = re.sub(
            r'(Check your email|Your order has been delivered|'
            r'Monthly invoice is ready|Sign-in successful)[^.]*\.?',
            '', t, flags=re.IGNORECASE
        )
        t = re.sub(r'http\S+|www\.\S+', '', t)
        t = re.sub(r'@\S+', '', t)
        t = re.sub(r'\S+@\S+', '', t)  # remove email addresses

        # --- Step 3: Expand contractions and map slang ---
        t = expand_contractions(t)
        t = apply_slang_map(t)

        # --- Step 4: Lowercase and strip non-alpha characters ---
        t = re.sub(r'[^a-zA-Z\s]', ' ', t).lower().strip()
        t = re.sub(r'\s+', ' ', t)  # collapse multiple spaces

        if not t:
            t = "notification"

        cleaned_texts.append(t)
        meta_features.append([
            exclamation_count, question_count,
            has_html_artifacts, is_all_caps,
            char_cnt, word_cnt
        ])

    df = df.copy()
    df['cleaned_text'] = cleaned_texts

    meta_cols = ['exclamation_count', 'question_count', 'has_html_artifacts',
                 'is_all_caps', 'char_cnt', 'word_cnt']
    for col in meta_cols:
        if col in df.columns:
            df = df.drop(columns=[col])

    meta_df = pd.DataFrame(meta_features, columns=meta_cols, index=df.index)
    return pd.concat([df, meta_df], axis=1)

# Apply to all three splits
train_df = clean_text(train_df)
val_df   = clean_text(val_df)
test_df  = clean_text(test_df)

print("Preprocessing done.")
print(f"Sample cleaned text: {train_df['cleaned_text'].iloc[0][:120]}")

## 3. Load GloVe Embeddings
We load the 100-dimensional GloVe vectors. These are used by both models — the Random Forest uses averaged GloVe vectors as document representations, and the GRU uses them to initialise its embedding layer.

In [ ]:
print("Loading GloVe 100d vectors...")
embeddings_lookup = {}
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        vector = np.asarray(parts[1:], dtype='float32')
        embeddings_lookup[word] = vector

print(f"Loaded {len(embeddings_lookup):,} word vectors.")

---
## 4. Random Forest Classifier
The Random Forest takes each email/message and represents it as a combination of:
1. A mean GloVe vector (100 dimensions) — captures semantic meaning
2. TF-IDF features (top 500) — captures important vocabulary patterns
3. Structural metadata — exclamation count, HTML presence, ALL CAPS, length

This hybrid feature set gives RF much more signal than text alone.

In [ ]:
# --- Build document vectors using mean GloVe pooling ---
def mean_glove_vector(texts, glove_dict, dim=100):
    vecs = []
    for text in texts:
        tokens = text.split()
        token_vecs = [glove_dict[w] for w in tokens if w in glove_dict]
        if token_vecs:
            vecs.append(np.mean(token_vecs, axis=0))
        else:
            vecs.append(np.zeros(dim))
    return np.array(vecs)

print("Building GloVe document vectors...")
X_train_glove = mean_glove_vector(train_df['cleaned_text'], embeddings_lookup)
X_val_glove   = mean_glove_vector(val_df['cleaned_text'],   embeddings_lookup)
X_test_glove  = mean_glove_vector(test_df['cleaned_text'],  embeddings_lookup)

# --- Build TF-IDF features (top 500 unigrams + bigrams) ---
# TF-IDF catches specific vocabulary patterns that GloVe averages out
print("Fitting TF-IDF vectorizer...")
tfidf = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)
X_train_tfidf = tfidf.fit_transform(train_df['cleaned_text']).toarray()
X_val_tfidf   = tfidf.transform(val_df['cleaned_text']).toarray()
X_test_tfidf  = tfidf.transform(test_df['cleaned_text']).toarray()

# --- Stack all features together ---
meta_cols = ['exclamation_count', 'question_count', 'has_html_artifacts',
             'is_all_caps', 'char_cnt', 'word_cnt']

X_train_rf = np.hstack([X_train_glove, X_train_tfidf, train_df[meta_cols].values])
X_val_rf   = np.hstack([X_val_glove,   X_val_tfidf,   val_df[meta_cols].values])
X_test_rf  = np.hstack([X_test_glove,  X_test_tfidf,  test_df[meta_cols].values])

y_train_rf = train_df['label'].values
y_val_rf   = val_df['label'].values
y_test_rf  = test_df['label'].values

print(f"Feature matrix shape — Train: {X_train_rf.shape}, Test: {X_test_rf.shape}")

In [ ]:
# --- Train the Random Forest ---
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=500,       # more trees = more stable predictions
    max_depth=20,           # slightly deeper trees to capture more patterns
    min_samples_leaf=2,     # prevents overfitting on noisy training samples
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_rf, y_train_rf)

# --- Evaluate ---
rf_train_preds = rf_model.predict(X_train_rf)
rf_val_preds   = rf_model.predict(X_val_rf)
rf_test_preds  = rf_model.predict(X_test_rf)

print("\n" + "="*50)
print("  RANDOM FOREST — Training Set")
print("="*50)
print(classification_report(y_train_rf, rf_train_preds, target_names=label_map.keys()))

print("\n" + "="*50)
print("  RANDOM FOREST — Validation Set")
print("="*50)
print(classification_report(y_val_rf, rf_val_preds, target_names=label_map.keys()))

print("\n" + "="*50)
print("  RANDOM FOREST — Cross-Domain Test Set (Gmail + WhatsApp)")
print("="*50)
print(classification_report(y_test_rf, rf_test_preds, target_names=label_map.keys()))

---
## 5. Bidirectional GRU
The GRU reads each message as a sequence of word embeddings. Key design choices:
- GloVe embeddings to initialise the embedding layer (kept trainable so it adapts)
- `SpatialDropout1D` to regularise against domain-specific vocabulary
- `clipnorm=1.0` on the optimiser to prevent gradient explosion on long emails
- Class weights to handle any remaining label imbalance
- `MAX_LEN=150` so longer emails are not cut off mid-sentence

In [ ]:
VOCAB_SIZE = 15000
MAX_LEN    = 150   # increased from 100 — emails need the extra length

# Fit tokenizer on training text only
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<UNK>")
tokenizer.fit_on_texts(train_df['cleaned_text'])

def tokenize_and_pad(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding='post', truncating='post')

X_train_gru = tokenize_and_pad(train_df['cleaned_text'])
X_val_gru   = tokenize_and_pad(val_df['cleaned_text'])
X_test_gru  = tokenize_and_pad(test_df['cleaned_text'])

y_train_gru = train_df['label'].values
y_val_gru   = val_df['label'].values
y_test_gru  = test_df['label'].values

# Build the GloVe embedding matrix for this vocabulary
embedding_matrix = np.zeros((VOCAB_SIZE, 100))
matched = 0
for word, idx in tokenizer.word_index.items():
    if idx < VOCAB_SIZE:
        vec = embeddings_lookup.get(word)
        if vec is not None:
            embedding_matrix[idx] = vec
            matched += 1

print(f"Vocabulary size: {VOCAB_SIZE:,}")
print(f"GloVe coverage: {matched:,} / {VOCAB_SIZE:,} tokens matched")

In [ ]:
# --- Build the GRU model ---
gru_model = Sequential([
    # Pre-trained GloVe embeddings, kept trainable so the model can adapt to email/WhatsApp language
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=100,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=True
    ),
    # SpatialDropout1D drops entire word embedding dimensions rather than individual neurons
    # This forces the model not to rely on any single word type (e.g. a common footer word)
    SpatialDropout1D(0.3),
    Bidirectional(GRU(
        64,
        dropout=0.2,
        recurrent_dropout=0.0,   # recurrent dropout disabled to keep training stable
        return_sequences=False
    )),
    Dropout(0.4),
    Dense(3, activation='softmax')  # 3 classes: Negative, Neutral, Positive
])

# clipnorm=1.0 stops long email sequences from causing gradient explosion
gru_model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    metrics=['accuracy']
)

gru_model.summary()

In [ ]:
# --- Compute class weights to handle any imbalance ---
classes       = np.unique(y_train_gru)
class_weights_arr = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_gru)
class_weights = dict(zip(classes, class_weights_arr))
print("Class weights:", class_weights)

# --- Callbacks ---
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=1, min_lr=1e-6)

# --- Train ---
print("\nTraining GRU...")
history = gru_model.fit(
    X_train_gru, y_train_gru,
    epochs=10,
    batch_size=64,
    validation_data=(X_val_gru, y_val_gru),
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# --- Evaluate GRU ---
gru_train_preds = np.argmax(gru_model.predict(X_train_gru, batch_size=64, verbose=0), axis=1)
gru_val_preds   = np.argmax(gru_model.predict(X_val_gru,   batch_size=64, verbose=0), axis=1)
gru_test_preds  = np.argmax(gru_model.predict(X_test_gru,  batch_size=64, verbose=0), axis=1)

print("\n" + "="*50)
print("  GRU — Training Set")
print("="*50)
print(classification_report(y_train_gru, gru_train_preds, target_names=label_map.keys()))

print("\n" + "="*50)
print("  GRU — Validation Set")
print("="*50)
print(classification_report(y_val_gru, gru_val_preds, target_names=label_map.keys()))

print("\n" + "="*50)
print("  GRU — Cross-Domain Test Set (Gmail + WhatsApp)")
print("="*50)
print(classification_report(y_test_gru, gru_test_preds, target_names=label_map.keys()))

## 6. GRU Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('GRU Accuracy per Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('GRU Loss per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Confusion Matrices — Cross-Domain Test Set
These show exactly where each model is making mistakes on the Gmail + WhatsApp data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_names = list(label_map.keys())

# Random Forest
rf_cm = confusion_matrix(y_test_rf, rf_test_preds)
sns.heatmap(rf_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('Random Forest — Test Confusion Matrix', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# GRU
gru_cm = confusion_matrix(y_test_gru, gru_test_preds)
sns.heatmap(gru_cm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('GRU — Test Confusion Matrix', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Final summary
rf_acc  = accuracy_score(y_test_rf,  rf_test_preds)
gru_acc = accuracy_score(y_test_gru, gru_test_preds)
print(f"\nRandom Forest Test Accuracy : {rf_acc:.4f}")
print(f"GRU Test Accuracy           : {gru_acc:.4f}")